In [0]:
%run ./00_Organizacao_do_Ambiente

bronze_schema: workspace.bronze
silver_schema: workspace.silver
gold_schema: workspace.gold
landing_path: /Volumes/workspace/rocket/cinedata_raw


In [0]:
# Cria a estrutura da arquitetura Medalhão (Bronze/Silver/Gold) já de uma vez;
# IF NOT EXISTS evita erro ao rodar o notebook mais de uma vez
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Schemas bronze, silver e gold criados.")

Schemas bronze, silver e gold criados.


In [0]:
from pyspark.sql.functions import current_timestamp

# Leitura pura + gravação com timestamp de ingestão.
# append: cada execução adiciona linhas, conforme exigência da atividade.
def carregar_bronze(nome_arquivo: str, nome_tabela: str):
    path = f"{landing_path}/{nome_arquivo}"
    df_raw = spark.read.csv(path, header=True, inferSchema=True)
    df_raw \
        .withColumn("ingestion_datetime", current_timestamp()) \
        .write.format("delta").mode("append") \
        .saveAsTable(f"{bronze_schema}.{nome_tabela}")
    print(f"{bronze_schema}.{nome_tabela} gravada com sucesso.")

# movies_info_TMDB_IMDB.csv: nome do arquivo veio com a ordem invertida (TMDB_IMDB) na origem
carregar_bronze("movies_info_TMDB_IMDB.csv", "tb_movies_info")
carregar_bronze("movies_financials_IMDB_TMDB.csv", "tb_movies_financials")
carregar_bronze("movies_metrics_IMDB_TMDB.csv", "tb_movies_metrics")
carregar_bronze("credits_and_tags_IMDB_TMDB.csv", "tb_credits_and_tags")
carregar_bronze("movies_reviews.csv", "tb_movies_reviews")

workspace.bronze.tb_movies_info gravada com sucesso.
workspace.bronze.tb_movies_financials gravada com sucesso.
workspace.bronze.tb_movies_metrics gravada com sucesso.
workspace.bronze.tb_credits_and_tags gravada com sucesso.
workspace.bronze.tb_movies_reviews gravada com sucesso.


In [0]:
from datetime import datetime, timedelta

# Sugestão da atividade: últimos 7 dias corridos, já que a API não cotiza fim de semana/feriado
data_fim_padrao = datetime.today().strftime("%m-%d-%Y")
data_inicio_padrao = (datetime.today() - timedelta(days=7)).strftime("%m-%d-%Y")

# Widgets permitem reexecutar o notebook com outro período sem editar o código
dbutils.widgets.text("data_inicio", data_inicio_padrao)
dbutils.widgets.text("data_fim", data_fim_padrao)

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")
print(f"Período: {data_inicio} até {data_fim}")

Período: 09-14-2026 até 09-21-2026


In [0]:
import requests
from pyspark.sql import Row

# Chama a API externa do Bacen pra obter a cotação do dólar no período informado
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

response = requests.get(url, timeout=15)
cotacao_json = response.json()["value"]

print(f"{len(cotacao_json)} cotações obtidas")

5 cotações obtidas


In [0]:
# Converte a lista de dicts (JSON) em Row para o Spark conseguir montar o DataFrame
rows = [Row(dataHoraCotacao=item["dataHoraCotacao"], cotacaoCompra=item["cotacaoCompra"]) for item in cotacao_json]
df_cotacao_raw = spark.createDataFrame(rows)

df_cotacao_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")

display(spark.table(f"{bronze_schema}.tb_cotacao_dolar"))

dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-14 13:10:08.144425,5.169,2026-09-21T15:11:54.139Z
2026-09-15 13:09:19.199664,5.1484,2026-09-21T15:11:54.139Z
2026-09-16 13:05:30.35873,5.152,2026-09-21T15:11:54.139Z
2026-09-17 13:03:21.858212,5.1515,2026-09-21T15:11:54.139Z
2026-09-18 13:03:34.742036,5.1569,2026-09-21T15:11:54.139Z


In [0]:
# Printa todas as tabelas criadas no schema bronze
display(spark.sql("SHOW TABLES IN workspace.bronze"))

database,tableName,isTemporary
bronze,tb_cotacao_dolar,false
bronze,tb_credits_and_tags,false
bronze,tb_movies_financials,false
bronze,tb_movies_info,false
bronze,tb_movies_metrics,false
bronze,tb_movies_reviews,false
